# DialectSentEval 2026, Subtask 2 - Supplementary Analysis

Team **Sabaa**. This notebook strengthens three claims the system paper makes
but does not currently measure directly.

**Part A** measures how accurately the source polarity can be inferred. The
paper estimates this at 0.803 by solving an equation; here it is measured
against gold labels on the development split, where `source_polarity` is
supplied. Runs in about two minutes and needs no trained model.

**Part B** uses that measurement to predict the official test score, which
tests the decomposition in §5.3 of the paper rather than assuming it.

## Setup

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets sacrebleu sentencepiece accelerate openpyxl

In [ ]:
"""Shared imports, paths, and the classifier used throughout."""

import glob
import os
import warnings

import numpy as np
import pandas as pd
import torch
from sacrebleu.metrics import CHRF
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoTokenizer

warnings.filterwarnings("ignore")


def find_file(filename):
    """Return the path to `filename`, searching the usual Kaggle and local roots."""
    for path in (f"/kaggle/input/{filename}", f"/workspace/{filename}",
                 f"/root/{filename}", f"./{filename}"):
        if os.path.exists(path):
            return path
    for root in ("/kaggle/input", "/workspace", "/root", "."):
        matches = glob.glob(f"{root}/**/{filename}", recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(filename)


WORK_DIR  = "/kaggle/working" if os.path.exists("/kaggle/working") else "/workspace"
CLF_MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
CLF_BATCH = 64

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


def load_sentiment_classifier():
    """Load CAMeLBERT-DA, neutralising the legacy-checkpoint guard if present."""
    import transformers.modeling_utils as modeling_utils

    guard = getattr(modeling_utils, "check_torch_load_is_safe", None)
    if guard is not None:
        modeling_utils.check_torch_load_is_safe = lambda: None
    try:
        tokenizer = AutoTokenizer.from_pretrained(CLF_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(
            CLF_MODEL, torch_dtype=torch.float32
        ).to(DEVICE)
    finally:
        if guard is not None:
            modeling_utils.check_torch_load_is_safe = guard
    model.eval()
    return tokenizer, model


clf_tokenizer, clf_model = load_sentiment_classifier()
LABEL_MAP    = clf_model.config.id2label
LABEL_TO_IDX = {v.capitalize(): k for k, v in LABEL_MAP.items()}


def softmax_probs(texts):
    """Return the classifier's probability vector for each text."""
    encoded = clf_tokenizer(
        texts, return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        return torch.softmax(clf_model(**encoded).logits, dim=-1).cpu().numpy()


def infer_source_polarity(texts):
    """Assign a definite Positive or Negative polarity, ignoring the Neutral class."""
    positive_idx = LABEL_TO_IDX["Positive"]
    negative_idx = LABEL_TO_IDX["Negative"]
    probs = softmax_probs(texts)
    return ["Positive" if p[positive_idx] >= p[negative_idx] else "Negative" for p in probs]


def classify_all(texts):
    """Argmax label and confidence for a long list, in classifier-sized batches."""
    results = []
    for start in range(0, len(texts), CLF_BATCH):
        for p in softmax_probs(texts[start:start + CLF_BATCH]):
            results.append({"label": LABEL_MAP[int(np.argmax(p))], "score": float(np.max(p))})
    return results


def invert_polarity(polarity):
    """Return the opposite polarity label."""
    return "Negative" if polarity == "Positive" else "Positive"


print("Classifier ready:", LABEL_MAP)

---

## Part A: How accurately can the source polarity be inferred?

The test set withholds `source_polarity`, so the system infers it. The
development set supplies it, which makes the inference step directly measurable
there. Whatever accuracy it achieves on development is the best available
estimate of its accuracy on test.

In [ ]:
val_df = pd.read_excel(find_file("SentimentSwapSharedTaskVal.xlsx"))

sources  = val_df["source"].astype(str).tolist()
gold     = val_df["source_polarity"].tolist()

inferred = []
for start in range(0, len(sources), CLF_BATCH):
    inferred.extend(infer_source_polarity(sources[start:start + CLF_BATCH]))

val_df["inferred_polarity"] = inferred

correct        = sum(g == i for g, i in zip(gold, inferred))
polarity_acc   = correct / len(val_df)

print(f"Source-polarity inference accuracy: {correct}/{len(val_df)} = {polarity_acc:.4f}")
print()
print("Confusion (rows = gold, columns = inferred):")
print(pd.crosstab(pd.Series(gold, name="gold"),
                  pd.Series(inferred, name="inferred")).to_string())
print()
for polarity in ("Positive", "Negative"):
    mask = [g == polarity for g in gold]
    hits = sum(i == polarity for i, m in zip(inferred, mask) if m)
    print(f"  Recall on {polarity:<8}: {hits}/{sum(mask)} = {hits / sum(mask):.4f}")

### Which sentences does it get wrong?

Inspecting the failures shows whether the errors are systematic, which matters
for the paper's recommendation about generating in both directions.

In [ ]:
errors = val_df[val_df["source_polarity"] != val_df["inferred_polarity"]]
print(f"{len(errors)} misclassified sources. First few:\n")

for _, row in errors.head(6).iterrows():
    print(f"  gold={row['source_polarity']:<8} inferred={row['inferred_polarity']:<8} "
          f"{str(row['source'])[:58]}")

error_words = errors["source"].astype(str).str.split().str.len()
all_words   = val_df["source"].astype(str).str.split().str.len()
print()
print(f"Mean length, misclassified : {error_words.mean():.1f} words")
print(f"Mean length, all           : {all_words.mean():.1f} words")

---

## Part B: Does the measured accuracy predict the official test score?

Section 5.3 of the paper decomposes the official sentiment score as

    official = p * h + (1 - p) * (1 - h)

where `h` is the measured rate at which outputs reach their intended target
polarity and `p` is the accuracy of polarity inference. The paper solves this
for `p`. Substituting the value measured in Part A instead predicts the official
score, so the two can be compared.

In [ ]:
HIT_RATE       = 0.9037   # measured in 02_final_evaluation.ipynb: 1745/1931
OFFICIAL_SCORE = 0.7447   # official test leaderboard

predicted = polarity_acc * HIT_RATE + (1 - polarity_acc) * (1 - HIT_RATE)
implied   = (OFFICIAL_SCORE - (1 - HIT_RATE)) / (HIT_RATE - (1 - HIT_RATE))

print("Decomposition check")
print("-" * 52)
print(f"  Measured polarity accuracy (dev, Part A) : {polarity_acc:.4f}")
print(f"  Implied by the official score            : {implied:.4f}")
print(f"  Difference                               : {abs(polarity_acc - implied):.4f}")
print()
print(f"  Predicted official score from measurement: {predicted:.4f}")
print(f"  Actual official score                    : {OFFICIAL_SCORE:.4f}")
print(f"  Prediction error                         : {abs(predicted - OFFICIAL_SCORE):.4f}")
print()
print("The two estimates of polarity accuracy are independent: one is measured")
print("on development against gold labels, the other is solved for from the test")
print("score. Agreement between them supports the decomposition; a large gap")
print("would indicate the test split is harder to classify than development,")
print("which is itself worth reporting.")